In [15]:
import pandas as pd

# Load all OTT datasets
netflix = pd.read_csv("netflix_titles.csv")
prime = pd.read_csv("amazonprime_titles.csv")
disney = pd.read_csv("Disney_titles.csv")
hbo = pd.read_csv("hbo_titles.csv")
apple = pd.read_csv("AppleTV_titles.csv")
paramount = pd.read_csv("paramount_titles.csv")

# Add available_on column
netflix["available_on"] = "Netflix"
prime["available_on"] = "Amazon Prime"
disney["available_on"] = "Disney+"
hbo["available_on"] = "HBO"
apple["available_on"] = "Apple TV"
paramount["available_on"] = "Paramount+"

# Combine datasets
all_data = pd.concat(
    [netflix, prime, disney, hbo, apple, paramount],
    ignore_index=True
)

# Clean title
all_data["title"] = all_data["title"].str.strip()

# Convert IMDb rating to numeric
all_data["imdb_score"] = pd.to_numeric(
    all_data["imdb_score"], errors="coerce"
)

# Group by title + release_year
grouped = all_data.groupby(["title", "release_year"], as_index=False)

# Aggregate properly
final_dataset = grouped.agg({
    "type": "first",
    "genres": "first",
    "description": "first",
    "imdb_score": "max",
    "available_on": lambda x: list(x.unique())
})

final_dataset.insert(0, "new_id", range(1, len(final_dataset) + 1))

# Save dataset
final_dataset.to_csv("multi_ott_dataset.csv", index=False)

print("New dataset with IMDb ratings created successfully!")

New dataset with IMDb ratings created successfully!


In [16]:
df=pd.read_csv("multi_ott_dataset.csv")

In [17]:
df[100:200]

,new_id,title,release_year,type,genres,description,imdb_score,available_on
100,101,137 Shots,2021,MOVIE,"['crime', 'documentation']","In this documentary, law enforcement faces scr...",6.4,['Netflix']
101,102,13: The Musical,2022,MOVIE,"['drama', 'family', 'comedy']","After his parents' divorce, Evan Goldman moves...",5.2,['Netflix']
102,103,13:14: The Challenge of Helping,2022,MOVIE,"['documentation', 'history']","On September 19, 2017, at 1:14 p.m., an earthq...",4.2,['Amazon Prime']
103,104,13th,2016,MOVIE,"['documentation', 'crime', 'history']",An in-depth look at the prison system in the U...,8.2,['Netflix']
104,105,14 Blades,2010,MOVIE,"['drama', 'history', 'thriller', 'action']",Commander Qinglong is the loyal leader of the ...,6.3,['Amazon Prime']
...,...,...,...,...,...,...,...,...
195,196,"3 ½ Minutes, 10 Bullets",2015,MOVIE,"['thriller', 'documentation', 'crime']","Black Friday, the day after Thanksgiving Novem...",7.2,['HBO']
196,197,3%,2016,SHOW,"['drama', 'scifi', 'thriller', 'action']",In a future where the elite inhabit an island ...,7.3,['Netflix']
197,198,30 Coins,2020,SHOW,"['thriller', 'drama', 'fantasy', 'horror', 'sc...",An exiled priest tries to escape his demons wh...,7.2,['HBO']
198,199,30 Day Promise,2017,MOVIE,['drama'],Dan and Heather Winslow are faced with the big...,7.7,['Amazon Prime']


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23319 entries, 0 to 23318
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   new_id        23319 non-null  int64  
 1   title         23319 non-null  object 
 2   release_year  23319 non-null  int64  
 3   type          23319 non-null  object 
 4   genres        23319 non-null  object 
 5   description   23124 non-null  object 
 6   imdb_score    20771 non-null  float64
 7   available_on  23319 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 1.4+ MB


In [24]:
df = df.dropna(subset=['description'])


df['imdb_score'] = df['imdb_score'].fillna(0.0)

df = df.reset_index(drop=True)

print(f"Cleaning Complete!")
print(f"Original Rows: 23,319")
print(f"Cleaned Rows: {len(df)}")

df.to_csv('cleaned_multi_ott_dataset.csv', index=False)

print("Balanced Dataset Created!")

/tmp/ipython-input-910196542.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['imdb_score'] = df['imdb_score'].fillna(0.0)


Cleaning Complete!
Original Rows: 23,319
Cleaned Rows: 23124
Balanced Dataset Created!


In [26]:
def clean_genres(genre_str):
    if isinstance(genre_str, str):
        # Remove brackets, quotes and commas
        return re.sub(r"[\[\]',]", " ", genre_str).lower()
    return ""

In [32]:
import re
df['genres_cleaned'] = df['genres'].apply(clean_genres)

In [33]:
#CombineFeatures(Description + Genres)
df['combined_features'] = df['description'].str.lower() + " " + df['genres_cleaned']

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23124 entries, 0 to 23123
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   new_id             23124 non-null  int64  
 1   title              23124 non-null  object 
 2   release_year       23124 non-null  int64  
 3   type               23124 non-null  object 
 4   genres             23124 non-null  object 
 5   description        23124 non-null  object 
 6   imdb_score         23124 non-null  float64
 7   available_on       23124 non-null  object 
 8   genres_cleaned     23124 non-null  object 
 9   combined_features  23124 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 1.8+ MB


In [39]:
df.head()

,new_id,title,release_year,type,genres,description,imdb_score,available_on,genres_cleaned,combined_features
0,1,#ABtalks,2018,SHOW,[],#ABtalks is a YouTube interview show hosted by...,9.6,['Netflix'],,#abtalks is a youtube interview show hosted by...
1,2,#Alive,2020,MOVIE,"['thriller', 'action', 'horror', 'drama']","As a grisly virus rampages a city, a lone man ...",6.3,['Netflix'],thriller action horror drama,"as a grisly virus rampages a city, a lone man ..."
2,3,#AnneFrank. Parallel Stories,2019,MOVIE,"['drama', 'history', 'documentation']",One single Anne Frank moves us more than the c...,6.5,['Netflix'],drama history documentation,one single anne frank moves us more than the c...
3,4,#FriendButMarried,2018,MOVIE,"['comedy', 'drama', 'romance']","Pining for his high school crush for years, a ...",6.8,['Netflix'],comedy drama romance,"pining for his high school crush for years, a ..."
4,5,#FriendButMarried 2,2020,MOVIE,"['drama', 'romance', 'comedy']",As Ayu and Ditto finally transition from best ...,6.9,['Netflix'],drama romance comedy,as ayu and ditto finally transition from best ...


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [42]:
#Vectorization
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

In [43]:
#Build the Model
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=10)
model_knn.fit(tfidf_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=10)

In [ ]:
import pickle

# Create a single dictionary to hold all components
recommendation_engine = {
    'movie_db': df,
    'vectorizer': tfidf,
    'matrix': tfidf_matrix,
    'model': model_knn
}

# Save the entire dictionary into one file
with open('ott_recommendation_engine.pkl', 'wb') as f:
    pickle.dump(recommendation_engine, f)

print(" saved in 'ott_recommendation_engine.pkl'")

 saved in 'ott_recommendation_engine.pkl'


In [ ]:
# --- STEP 2: THE AGGREGATOR ENGINE ---
def get_hybrid_recommendations(target_title, target_ott, target_type, sim_weight=0.7, imdb_weight=0.3):
    try:
        # 1. Locate movie
        idx = df[df['title'].str.lower() == target_title.lower()].index[0]
        actual_title = df['title'].iloc[idx]
        
        # 2. Get large pool
        distances, indices = model_knn.kneighbors(tfidf_matrix[idx], n_neighbors=200)
        indices = indices.flatten()
        distances = distances.flatten()
        
        candidates_map = {} # Key: Title, Value: Dictionary of movie info

        for i in range(1, len(indices)):
            res_idx = indices[i]
            title = df['title'].iloc[res_idx]
            ott_raw = str(df['available_on'].iloc[res_idx])
            ott_clean = re.sub(r"[\[\]']", "", ott_raw)

            # --- DEDUPLICATION & APPENDING LOGIC ---
            if title.lower() in candidates_map:
                # If we've seen it, just add the new OTT platform to the existing string
                existing_ott = candidates_map[title.lower()]['OTT']
                if ott_clean not in existing_ott:
                    candidates_map[title.lower()]['OTT'] += f", {ott_clean}"
                continue # Move to next neighbor (don't create a new recommendation)

            # --- TYPE FILTERING ---
            item_type = df['type'].iloc[res_idx]
            if target_type != "Both" and item_type.lower() != target_type.lower():
                continue

            # --- PLATFORM FILTERING ---
            # If user selected a specific OTT, only start the entry if it's there
            if target_ott == "All" or target_ott.lower() in ott_raw.lower():
                sim_score = 1 - distances[i]
                imdb_val = df['imdb_score'].iloc[res_idx]
                hybrid_score = (sim_score * sim_weight) + ((imdb_val / 10) * imdb_weight)
                
                candidates_map[title.lower()] = {
                    "Title": title,
                    "Year": df['release_year'].iloc[res_idx],
                    "Type": item_type,
                    "Score": imdb_val if imdb_val > 0 else "-",
                    "Match": f"{sim_score * 100:.1f}%",
                    "RankScore": hybrid_score,
                    "OTT": ott_clean,
                    "Desc": df['description'].iloc[res_idx]
                }

        # 3. Sort the unique items by RankScore and take Top 5
        final_list = sorted(candidates_map.values(), key=lambda x: x['RankScore'], reverse=True)[:5]
            
        # --- OUTPUT ---
        if final_list:
            print(f"\n[ENGINE] Top {len(final_list)} Unique Matches for '{actual_title}':")
            print("=" * 115)
            for i, item in enumerate(final_list, 1):
                year_display = f"({int(item['Year'])})" if pd.notnull(item['Year']) else ""
                print(f"{i}. {item['Title']} {year_display} | IMDb: {item['Score']} | Similarity: {item['Match']}")
                print(f"   Available on: {item['OTT']}")
                print(f"   Summary: {item['Desc']}\n")
                print("-" * 115)
        else:
            print(f"\nNo results found for {target_ott}.")

    except IndexError:
        print(f"\n'{target_title}' not found.")

In [66]:
# --- UPDATED EXECUTION ---
print("Welcome to the OTT Recommender!")
user_movie = input("Enter a movie/show you liked: ") 
user_ott = input("Search in (Netflix, Amazon Prime, Disney+, or All): ")

user_type = input("What are you looking for? (Movie, Show, or Both): ").strip()

get_hybrid_recommendations(user_movie, user_ott, user_type)

Welcome to the OTT Recommender!

[ENGINE] Top 5 Unique Matches for 'The Dark Knight Rises':
1. Batman: The Long Halloween, Part One (2021) | IMDb: 7.2 | Similarity: 40.1%
   Available on: HBO
   Summary: Following a brutal series of murders taking place on Halloween, Thanksgiving, and Christmas, Gotham City's young vigilante known as the Batman sets out to pursue the mysterious serial killer alongside police officer James Gordon and district attorney Harvey Dent.

-------------------------------------------------------------------------------------------------------------------
2. Batman: The Long Halloween, Part Two (2021) | IMDb: 7.2 | Similarity: 39.9%
   Available on: HBO
   Summary: As Gotham City's young vigilante, the Batman, struggles to pursue a brutal serial killer, district attorney Harvey Dent gets caught in a feud involving the criminal family of the Falcones.

-----------------------------------------------------------------------------------------------------------------